# 03 Check WLASL2000 Keypoints — Strict Quality Control

## What this notebook does
This notebook validates WLASL2000 keypoints before final model training.

It checks:
- missing `.npy` files
- incorrect keypoint shapes
- high zero-ratio samples
- low-sample classes
- final clean training index

## Main output
```text
data/processed/ASL/WLASL2000/wlasl2000_clean_keypoint_index.csv
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

## 1. Set paths

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

DATASET_NAME = "WLASL2000"
PREFIX = "wlasl2000"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
KEYPOINT_DIR = BASE_DIR / "keypoints"

VIDEO_INDEX_FILE = BASE_DIR / f"{PREFIX}_video_index.csv"
KEYPOINT_INDEX_FILE = BASE_DIR / f"{PREFIX}_keypoint_index.csv"
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"
QUALITY_REPORT_FILE = BASE_DIR / f"{PREFIX}_keypoint_quality_report.csv"
LOW_SAMPLE_CLASS_FILE = BASE_DIR / f"{PREFIX}_low_sample_classes.csv"

video_df = pd.read_csv(VIDEO_INDEX_FILE)

print("Videos in index:", len(video_df))
print("Classes in video index:", video_df["label_id"].nunique())
print("Extracted .npy files:", len(list(KEYPOINT_DIR.glob("*.npy"))))

## 2. Rebuild keypoint index from actual files

In [ ]:
records = []

for _, row in video_df.iterrows():
    keypoint_path = KEYPOINT_DIR / f"{row['video_id']}.npy"

    if keypoint_path.exists():
        records.append({
            "video_id": row["video_id"],
            "gloss": row["gloss"],
            "label_id": int(row["label_id"]),
            "original_class_id": int(row["original_class_id"]),
            "video_path": row["video_path"],
            "keypoint_path": str(keypoint_path)
        })

keypoint_df = pd.DataFrame(records)
keypoint_df.to_csv(KEYPOINT_INDEX_FILE, index=False)

print("Saved keypoint index:", KEYPOINT_INDEX_FILE)
print("Keypoint samples:", len(keypoint_df))
print("Classes:", keypoint_df["label_id"].nunique())

keypoint_df.head()

## 3. Check missing keypoint files

In [ ]:
expected_ids = set(video_df["video_id"].astype(str))
actual_ids = set([p.stem for p in KEYPOINT_DIR.glob("*.npy")])

missing_ids = expected_ids - actual_ids

print("Missing keypoint files:", len(missing_ids))
list(missing_ids)[:20]

## 4. Check keypoint shape and zero ratio

In [ ]:
quality_records = []

for _, row in tqdm(keypoint_df.iterrows(), total=len(keypoint_df), desc="Checking keypoint quality"):
    arr = np.load(row["keypoint_path"])

    quality_records.append({
        "video_id": row["video_id"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "shape": str(arr.shape),
        "num_frames": arr.shape[0] if len(arr.shape) >= 1 else None,
        "num_features": arr.shape[1] if len(arr.shape) == 2 else None,
        "zero_ratio": float(np.mean(arr == 0)),
        "mean_abs_value": float(np.mean(np.abs(arr))),
        "std_value": float(np.std(arr))
    })

quality_df = pd.DataFrame(quality_records)
quality_df.to_csv(QUALITY_REPORT_FILE, index=False)

print("Saved quality report:", QUALITY_REPORT_FILE)
print("\nUnique shapes:")
print(quality_df["shape"].value_counts())

print("\nZero ratio summary:")
print(quality_df["zero_ratio"].describe())

quality_df.sort_values("zero_ratio", ascending=False).head(20)

## 5. Create clean keypoint index

In [ ]:
ZERO_RATIO_THRESHOLD = 0.95

valid_ids = set(
    quality_df[
        (quality_df["num_frames"] == 60) &
        (quality_df["num_features"] == 258) &
        (quality_df["zero_ratio"] < ZERO_RATIO_THRESHOLD)
    ]["video_id"].astype(str)
)

clean_df = keypoint_df[keypoint_df["video_id"].astype(str).isin(valid_ids)].copy()

zero_lookup = quality_df.set_index(quality_df["video_id"].astype(str))["zero_ratio"].to_dict()
clean_df["zero_ratio"] = clean_df["video_id"].astype(str).map(zero_lookup)

clean_df.to_csv(CLEAN_INDEX_FILE, index=False)

print("Saved clean index:", CLEAN_INDEX_FILE)
print("Clean samples:", len(clean_df))
print("Clean classes:", clean_df["label_id"].nunique())

clean_df.head()

## 6. Create low-sample class report

In [ ]:
class_counts = clean_df["gloss"].value_counts()

low_sample_df = class_counts.reset_index()
low_sample_df.columns = ["gloss", "clean_samples"]
low_sample_df = low_sample_df[low_sample_df["clean_samples"] < 3]

low_sample_df.to_csv(LOW_SAMPLE_CLASS_FILE, index=False)

print("Class distribution after cleaning")
print("---------------------------------")
print("Classes:", len(class_counts))
print("Minimum samples per class:", class_counts.min())
print("Maximum samples per class:", class_counts.max())
print("Average samples per class:", round(class_counts.mean(), 2))
print("Classes with fewer than 3 samples:", len(low_sample_df))
print("Saved low-sample class report:", LOW_SAMPLE_CLASS_FILE)

display(low_sample_df.head(30))

class_counts.head(30)

## 7. Plot class distribution

In [ ]:
plt.figure(figsize=(20, 5))
class_counts.plot(kind="bar")
plt.title("WLASL2000 Clean Samples per Class")
plt.xlabel("Gloss")
plt.ylabel("Number of Samples")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## Next notebook
Continue with:

```text
04_train_wlasl2000_light_v2_from_scratch.ipynb
```